# 🧪 Citadel Developer VPN - Deployment and Validation

## Overview

**Purpose:** give the developer machine private IP connectivity and private DNS resolution to an existing Citadel Governance Hub from its browser, local VS Code notebooks and terminal. This is a persistent, optional connectivity add-on, not a change to the primary accelerator deployment.

| Scenario | Configuration | Expected outcome |
|---|---|---|
| Isolated connectivity infrastructure | New dedicated resource group and hub VNet | No primary VNet address-space/subnet changes |
| Developer authentication | P2S OpenVPN + Microsoft Entra ID | Azure VPN Client on Windows/macOS |
| Private IP routing | Two peerings with gateway transit | P2S clients reach the primary VNet |
| Private DNS | Static inbound DNS Private Resolver + optional existing-zone links | Primary service FQDNs resolve to private IPs |
| End-to-end checks | Direct DNS, OS DNS, TCP 443 and trusted TLS | Network connectivity proven without claiming application authorization |

```text
Developer Windows/macOS machine (browser / local VS Code kernel / terminal)
  -> Azure VPN Client (Entra ID, OpenVPN, split tunnel)
  -> dedicated VPN resource group: hub VNet + VPN Gateway
       + DNS Private Resolver, static inbound endpoint
       + gateway-transit peering <-> existing governance primary VNet
       + resolution-only links from existing primary private DNS zones
```

## Expected outcomes

- Generated, reviewable Bicep parameters, Azure validate/what-if and an explicitly gated subscription deployment.
- Non-overlapping hub/client CIDRs checked against the actual primary VNet, directly peered VNets and supplied local/corporate ranges.
- Existing private DNS zones and records reused, never recreated. Zone discovery includes primary-RG zones, zones linked to the primary VNet and primary private-endpoint zone groups; additional full IDs cover external zones.
- Platform-specific VPN profiles and local validation after the user imports the profile and signs in.
- Optional cleanup removes only the owned VPN resource group and this add-on's external peering/DNS links.

## Azure Prerequisites

- A deployed accelerator and its actual primary VNet ID; BYO VNets in another RG/subscription are supported in the same Entra tenant.
- Azure **Public cloud**; Azure CLI with Bicep installed, `az login`, the intended subscription already selected, and an active deployed azd environment. Set `init_from_azd=False` only for explicit manual primary coordinates. No subscription switching or package installation is performed here.
- Subscription-level deployment/RG creation permissions; network write/peer permissions on both VNets; private DNS link write and VNet join permissions in every zone scope. The `Microsoft.Network` provider must be registered. Read permission across the subscriptions scanned for linked zones is needed; discovery errors are fatal, not silently omitted.
- A region supporting DNS Private Resolver and the selected VPN SKU, sufficient public IP/gateway quota, and approval for **ongoing VPN Gateway, Standard public IP, resolver inbound endpoint and peering charges**. Gateway creation/deletion can take 45 minutes or longer. Check [VPN pricing](https://azure.microsoft.com/pricing/details/vpn-gateway/) and [DNS pricing](https://azure.microsoft.com/pricing/details/dns/).
- Local Windows/macOS kernel on the developer machine. A remote SSH/container/Codespaces kernel has its own routing/DNS and does not inherit the host VPN automatically.
- Latest supported Azure VPN Client. **Linux is unsupported with this Entra-only design:** Microsoft retired its Linux client on August 31, 2026; supported OpenVPN/strongSwan Linux replacements require certificate/RADIUS authentication, outside this notebook's selected scope.

The primary VNet cannot already have a gateway/route server or consume another remote gateway. VNet peering is not transitive: this add-on grants access to the primary VNet, not all of its other peers. Review NSGs, UDRs, firewalls and private-endpoint network policies for the P2S pool. Nothing here opens public access or changes those controls.

## Microsoft sources and design decisions

- [Entra P2S gateway configuration](https://learn.microsoft.com/azure/vpn-gateway/point-to-site-entra-gateway): route-based, non-Basic gateway, OpenVPN, Microsoft-registered audience, correct tenant/issuer URLs.
- [Gateway-transit peering](https://learn.microsoft.com/azure/vpn-gateway/vpn-gateway-peering-gateway-transit): hub allows gateway transit; primary uses remote gateways; both allow forwarded traffic. Regenerate profiles after topology changes.
- [DNS resolver endpoints](https://learn.microsoft.com/azure/dns/private-resolver-endpoints-rulesets) and [resolver constraints](https://learn.microsoft.com/azure/dns/dns-private-resolver-overview): dedicated delegated /28 inbound subnet, static non-reserved IP, resolver in the same region as its VNet; linked zones resolve through inbound. No outbound endpoint/ruleset is needed for Azure private-zone resolution alone.
- [VPN client DNS and routing](https://learn.microsoft.com/azure/vpn-gateway/azure-vpn-client-optional-configurations): configure DNS servers/routes in the downloaded XML. Entra on Windows uses NRPT, not necessarily `ipconfig /all`; macOS DNS suffixes do not persist, so its profile omits them.
- [Linux client retirement](https://learn.microsoft.com/azure/vpn-gateway/azure-vpn-client-linux-retirement): Entra authentication is not supported by the replacement Linux clients.
- [VPN Gateway tutorial](https://learn.microsoft.com/azure/vpn-gateway/tutorial-create-gateway-portal): GatewaySubnet, AZ SKUs, Standard static public IP and deployment timing.

Infrastructure: [main.bicep](../bicep/infra/developer-vpn/main.bicep), [main.bicepparam](../bicep/infra/developer-vpn/main.bicepparam), [deployment notes](../bicep/infra/developer-vpn/README.md). Local generated artifacts remain under ignored `.runs/developer-vpn/`; do not commit generated profiles or notebook outputs.


<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

**Input:** the active azd environment plus the VPN defaults below. **Output:** resolved primary/VPN configuration; no Azure changes. The default configuration runs through validation/what-if without editing placeholders.

`None` means automatic discovery/defaults. Subscription, primary RG, region and APIM name are read once from the **active azd environment**. The VPN RG defaults to `<primary-rg>-vpn`; resource names use the `citadel-vpn` prefix. Explicit values always win. Set `init_from_azd=False` only when supplying primary coordinates manually. Missing or invalid azd configuration is an error, not a silent fallback to another environment.

The primary VNet is discovered from the primary RG, APIM subnet or primary private endpoints; an explicit ID is only needed for ambiguous topologies. Active local/corporate private IPv4 routes are detected automatically on Windows/macOS (Linux can prepare profiles but cannot use this Entra-only VPN). Override `local_network_cidrs` to add organizational ranges not visible in active routes. **Run preflight disconnected from this developer VPN** so its own installed routes do not appear as address conflicts; after connecting, run the local validation cell directly.

Choose `public_ip_zones = ['1', '2', '3']` only in a region supporting those zones; `[]` requests a regional Standard public IP. CIDRs shown below are examples, not assumed free ranges. `deploy_dns_resolver=False` requires a reachable existing DNS resolver; `link_private_dns_zones=False` leaves zone links untouched and makes DNS your responsibility.


In [ ]:
import hashlib
import ipaddress
import json
from pathlib import Path
import sys

repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'shared' / 'utils.py').is_file()), None)
if repo_root is None:
    raise RuntimeError('Open this notebook from the repository root or validation-local directory.')
sys.path.insert(0, str(repo_root / 'shared'))
import utils
import vpntools as vpn

# 🔧 GOVERNANCE HUB
init_from_azd = True
expected_subscription_id = None
governance_hub_resource_group = None
primary_vnet_id = None
apim_name = None
location = None

# 🔧 API VERSION: infrastructure only; no inference API calls.
use_inference_api = False
# 🧠 MODEL / 🔐 KEY VAULT / 🤖 MICROSOFT FOUNDRY: unused for connectivity.
use_model = False
use_keyvault_integration = False
use_foundry_integration = False

# VPN HUB AND CLIENT NETWORKS
vpn_resource_group = None  # <primary-rg>-vpn; set a string to override.
resource_prefix = None  # Defaults to citadel-vpn.
hub_address_prefix = '10.171.0.0/24'
client_address_prefix = '172.30.240.0/24'
local_network_cidrs = None  # Detect active private IPv4 routes; or provide an explicit list.
gateway_sku = 'VpnGw1AZ'
public_ip_zones = []
entra_audience = 'c632b3df-fb67-4d84-bdcf-b95ad541b5c8'

# PRIVATE DNS
deploy_dns_resolver = True
link_private_dns_zones = True
extra_private_dns_zone_ids = []
external_dns_server_ip = None  # Required only when deploy_dns_resolver=False.
dns_suffixes = ['.azure-api.net', '.openai.azure.com', '.services.ai.azure.com', '.cognitiveservices.azure.com', '.vault.azure.net', '.vaultcore.azure.net', '.blob.core.windows.net', '.file.core.windows.net', '.queue.core.windows.net', '.table.core.windows.net', '.documents.azure.com', '.search.windows.net', '.azurewebsites.net', '.servicebus.windows.net', '.redis.azure.net', '.monitor.azure.com']

# DELIBERATE STAGE GATES: review cost/what-if, then connect the desktop VPN client.
deploy_enabled = True
download_client_profiles = True
run_local_validation = True
validation_targets = None  # Discover primary private endpoints/APIM; or supply a host/expected_ips list.

defaults = vpn.notebook_defaults({
    'expected_subscription_id': expected_subscription_id,
    'governance_hub_resource_group': governance_hub_resource_group,
    'primary_vnet_id': primary_vnet_id,
    'apim_name': apim_name,
    'location': location,
    'vpn_resource_group': vpn_resource_group,
    'resource_prefix': resource_prefix,
}, from_azd=init_from_azd)
expected_subscription_id = defaults['expected_subscription_id']
governance_hub_resource_group = defaults['governance_hub_resource_group']
primary_vnet_id = defaults['primary_vnet_id']
apim_name = defaults['apim_name']
location = defaults['location']
vpn_resource_group = defaults['vpn_resource_group']
resource_prefix = defaults['resource_prefix']

outputs = None
azure_verified = False
local_results = []
profiles = []
utils.print_ok('Variables resolved; deployment and local validation remain explicitly gated.', defaults)


<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

**Input:** existing CLI login and expected subscription. **Output:** verified subscription/tenant. A mismatch fails rather than changing the active subscription. This runbook is Azure Public cloud only.


In [ ]:
with utils.notebook_step('Verify Azure CLI and subscription'):
    vpn.require_values(expected_subscription_id=expected_subscription_id, governance_hub_resource_group=governance_hub_resource_group, vpn_resource_group=vpn_resource_group)
    account = vpn.az_json('account', 'show')
    if account['id'].lower() != expected_subscription_id.lower():
        raise ValueError('Active CLI subscription differs from expected_subscription_id. Select the intended subscription outside this notebook.')
    if vpn.az_json('cloud', 'show')['name'] != 'AzureCloud':
        raise ValueError('This Entra configuration supports Azure Public cloud only.')
    subscription_id, tenant_id = account['id'], account['tenantId']
    provider = vpn.az_json('provider', 'show', '--namespace', 'Microsoft.Network', subscription=subscription_id)
    if provider['registrationState'] != 'Registered':
        raise ValueError('Register Microsoft.Network in the target subscription before continuing.')
    utils.print_info(f"Subscription: {account['name']} ({subscription_id}); tenant: {tenant_id}")


<a id='2'></a>
### 2️⃣ Discover Primary VNet and Check Transit Readiness

**Input:** primary RG/VNet, proposed VPN RG and prefix. **Output:** actual Azure address spaces and transit conflicts.

**Bootstrap exception to the APIM notebook convention:** initialize via ARM, not APIMClientTool. APIM may intentionally be private and unreachable until this notebook establishes the VPN. No APIM products, contracts, models or Foundry connections are created or changed.


In [ ]:
with utils.notebook_step('Discover primary VNet and transit readiness'):
    primary_vnet, azure_prefixes = vpn.inspect_primary(subscription_id, governance_hub_resource_group, primary_vnet_id, vpn_resource_group, resource_prefix, apim_name)
    primary_vnet_id = primary_vnet['id']
    if location is None or location == 'REPLACE':
        location = primary_vnet['location']
    resolved_local_network_cidrs = vpn.discover_local_networks() if local_network_cidrs is None else local_network_cidrs
    network_plan = vpn.validate_networks(hub_address_prefix, client_address_prefix, azure_prefixes, resolved_local_network_cidrs)
    dns_server_ip = network_plan['resolverIp'] if deploy_dns_resolver else external_dns_server_ip
    vpn.require_values(location=location, dns_server_ip=dns_server_ip)
    if not ipaddress.ip_address(dns_server_ip).is_private:
        raise ValueError('Use a reachable private DNS resolver IP, not a public DNS server.')
    utils.print_info(f"Primary VNet: {primary_vnet_id}; Azure CIDRs: {azure_prefixes}")
    utils.print_ok('Address ranges and gateway-transit prerequisites checked.', network_plan)
    utils.print_warning('Review primary NSGs/UDRs/firewalls for the client pool. Peering is not transitive; other primary peers are not automatically reachable.')


<a id='3'></a>
### 3️⃣ Discover Existing Private DNS Zones

**Input:** primary resources and optional extra zone IDs. **Output:** complete discovered zone inventory for review. Disable linking explicitly to opt out. A failed discovery is not treated as an empty successful result. DNS peering alone does **not** make the primary private zones visible to a hub resolver.

Primary-owned zones (even currently unused ones), primary-linked zones, and private-endpoint zone-group references are included. For external zones outside these visible scopes, supply full IDs. Inspect the printed inventory before deployment. Add the **original service FQDN's public suffix** to `dns_suffixes` for custom/private service domains; only adding a `privatelink.*` suffix can miss the original public-name lookup on Windows. The macOS profile uses the DNS server without suffix entries because of the documented client limitation.


In [ ]:
with utils.notebook_step('Discover private DNS zones'):
    zone_ids = vpn.discover_zones(subscription_id, governance_hub_resource_group, primary_vnet, extra_private_dns_zone_ids) if link_private_dns_zones else []
    if zone_ids:
        utils.print_ok(f'{len(zone_ids)} existing private DNS zones selected.', zone_ids)
    else:
        utils.print_warning('Private DNS zone linking is disabled. Configure equivalent resolution on the selected resolver yourself.')
    utils.print_info('Private validation targets are discovered from primary endpoint DNS/NIC settings and APIM when local validation is enabled. Set validation_targets only to override discovery.')


<a id='4'></a>
### 4️⃣ Generate Bicep Parameters, Validate and Preview

**Input:** the discovered network/DNS plan. **Output:** generated `.bicepparam`, compiled ARM template/parameters, Azure validation and saved what-if. These operations do not provision resources. Bicep is compiled once; validation, what-if and deployment consume `main.json` plus `@parameters.json`. This prevents older Azure CLI versions from writing Bicep installation status text ahead of the JSON result. Validation must exit successfully; what-if must return valid JSON with a successful status. Review external primary peering/DNS link changes as well as the new VPN RG. Local artifacts are deliberately excluded from Git.

The /27 `GatewaySubnet` starts at the hub base, the dedicated /28 inbound resolver subnet at base + 32, and its static DNS IP at base + 37. Bicep `cidrHost(subnet, 4)` means subnet base + 5, not + 4 ([Microsoft CIDR reference](https://learn.microsoft.com/azure/azure-resource-manager/bicep/bicep-functions-cidr#cidrhost)). After deployment, profiles and validation use the **actual deployment output**, never the preflight prediction. No NSG/UDR is added to GatewaySubnet. The hub retains Azure-provided DNS; only the VPN client uses the inbound resolver IP, avoiding a resolver loop.


In [ ]:
previewed_parameters = None
previewed_artifacts = {}
with utils.notebook_step('Generate parameters and preview Azure deployment'):
    template = repo_root / 'bicep' / 'infra' / 'developer-vpn' / 'main.bicep'
    artifact_dir = repo_root / 'validation-local' / '.runs' / 'developer-vpn' / vpn_resource_group
    compiled_template = artifact_dir / 'main.json'
    compiled_parameters = artifact_dir / 'parameters.json'
    parameters = {'vpnResourceGroupName': vpn_resource_group, 'location': location, 'primaryVnetId': primary_vnet_id, 'resourcePrefix': resource_prefix, 'hubAddressPrefix': hub_address_prefix, 'clientAddressPrefix': client_address_prefix, 'tenantId': tenant_id, 'audience': entra_audience, 'gatewaySku': gateway_sku, 'publicIpZones': public_ip_zones, 'deployDnsResolver': deploy_dns_resolver, 'linkPrivateDnsZones': link_private_dns_zones, 'privateDnsZoneIds': zone_ids}
    params_file = vpn.write_parameters(template, artifact_dir / 'main.bicepparam', parameters)
    vpn.az_json('bicep', 'build', '--file', str(template), '--outfile', str(compiled_template), expect_json=False)
    vpn.az_json('bicep', 'build-params', '--file', str(params_file), '--outfile', str(compiled_parameters), expect_json=False)
    deployment_name = 'developer-vpn-' + vpn_resource_group[:32] + '-' + hashlib.sha256(vpn_resource_group.lower().encode()).hexdigest()[:8]
    deployment_args = ['--name', deployment_name, '--location', location, '--template-file', str(compiled_template), '--parameters', '@' + str(compiled_parameters)]
    vpn.az_json('deployment', 'sub', 'validate', *deployment_args, subscription=subscription_id, timeout=900, expect_json=False)
    what_if = vpn.az_json('deployment', 'sub', 'what-if', *deployment_args, '--no-pretty-print', subscription=subscription_id, timeout=900)
    if what_if.get('status') != 'Succeeded' or what_if.get('error'):
        raise RuntimeError(f"Azure what-if did not succeed: {what_if}")
    (artifact_dir / 'what-if.json').write_text(json.dumps(what_if, indent=2), encoding='utf-8')
    previewed_parameters = params_file.read_text(encoding='utf-8')
    previewed_artifacts = {path: hashlib.sha256(path.read_bytes()).hexdigest() for path in (compiled_template, compiled_parameters)}
    utils.print_ok(f'Parameters and what-if saved in {artifact_dir}', what_if)
    utils.print_warning('Review ongoing cost and all what-if changes before enabling deployment. Partial failed deployments still incur charges.')


<a id='5'></a>
### 5️⃣ Deploy or Reopen This VPN Deployment

**Input:** reviewed what-if and `deploy_enabled=True` to provision. **Output:** persistent deployment outputs. Deployment uses the same compiled ARM files as what-if; changing either compiled file or the source parameters invalidates the preview. For an already reviewed plan, run `deploy_enabled = True` in a cell before this step; there is no need to rerun initialization just to change this gate. A disabled gate can reopen a previous deployment of the **same configuration** without creating resources.

**Retry after a failed deployment:** once a missing subscription feature is registered, rerun this step with the same reviewed plan. If initialization was rerun and reset the automatically discovered primary VNet ID, this step restores it from the fingerprint-checked compiled parameters, not from guessed defaults or resource tags. An explicitly changed primary VNet, VPN RG or configured subscription requires a new preview. Existing partial resources must still pass ownership checks; do not delete or retag the primary resources to work around an error. After updating shared helpers, run `import importlib; importlib.reload(vpn)` in a cell to refresh the helper without discarding the reviewed plan.

Changing names, CIDRs or zone selections does not remove old resources in incremental mode. Clean up the old add-on first when removing links or renaming resources. Keep the primary deployment intact. Do not rerun or modify the primary VNet while the add-on is deploying.


In [ ]:
outputs = None
azure_verified = False
local_results = []
with utils.notebook_step('Deploy or reopen developer VPN'):
    if (params_file.read_text(encoding='utf-8') != previewed_parameters or not previewed_artifacts
            or any(not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected
                   for path, expected in previewed_artifacts.items())):
        raise ValueError('Deployment inputs changed after preview; rerun validation and what-if.')
    reviewed_parameters = json.loads(compiled_parameters.read_text(encoding='utf-8'))['parameters']
    reviewed_primary_vnet_id = reviewed_parameters['primaryVnetId']['value']
    vpn.require_values(reviewed_primary_vnet_id=reviewed_primary_vnet_id)
    if primary_vnet_id not in (None, '', 'REPLACE') and primary_vnet_id.lower() != reviewed_primary_vnet_id.lower():
        raise ValueError('primary_vnet_id changed after preview; rerun discovery and what-if.')
    if vpn_resource_group.lower() != reviewed_parameters['vpnResourceGroupName']['value'].lower():
        raise ValueError('VPN resource group changed after preview; rerun discovery and what-if.')
    if expected_subscription_id.lower() != subscription_id.lower():
        raise ValueError('Configured subscription changed after preview; rerun subscription verification and what-if.')
    if primary_vnet_id in (None, '', 'REPLACE'):
        utils.print_info('Restoring the primary VNet ID from the unchanged, reviewed Step 4 deployment parameters.')
    primary_vnet_id = reviewed_primary_vnet_id
    parameters = {name: entry['value'] for name, entry in reviewed_parameters.items()}
    group_exists = vpn.check_group(subscription_id, vpn_resource_group, primary_vnet_id)
    if deploy_enabled:
        deployment = vpn.az_json('deployment', 'sub', 'create', *deployment_args, subscription=subscription_id, timeout=7200)
    elif group_exists:
        deployment = vpn.az_json('deployment', 'sub', 'show', '--name', deployment_name, subscription=subscription_id)
        recorded = deployment['properties']['parameters']
        if any(recorded.get(name, {}).get('value') != value for name, value in parameters.items()):
            raise ValueError('Existing deployment parameters differ from this plan; review and redeploy explicitly.')
    else:
        deployment = None
        utils.print_warning('Deployment disabled and no owned VPN RG found. Azure verification/profile/local checks will remain pending.')
    if deployment is not None:
        if deployment['properties']['provisioningState'] != 'Succeeded':
            raise RuntimeError('Deployment has not succeeded. Inspect deployment operations; do not assume partially created resources are free.')
        outputs = deployment['properties']['outputs']
        (artifact_dir / 'outputs.json').write_text(json.dumps(outputs, indent=2), encoding='utf-8')
        utils.print_ok('Persistent VPN deployment loaded.', outputs)


<a id='6'></a>
### 6️⃣ Verify Azure Resources and Transit

**Input:** completed deployment outputs. **Output:** gateway success, both Connected peerings with correct transit flags, static resolver IP and completed resolution-only DNS links. This is control-plane validation, not yet proof that the developer machine can reach a service.


In [ ]:
azure_verified = False
with utils.notebook_step('Verify Azure network resources'):
    if outputs is None:
        utils.print_warning('PENDING: deploy or load the owned VPN deployment first.')
    else:
        vpn.verify_azure_deployment(subscription_id, primary_vnet_id, outputs)
        dns_server_ip = vpn.client_dns_server(outputs, external_dns_server_ip)
        azure_verified = True


<a id='7'></a>
### 7️⃣ Generate and Customize VPN Client Profiles

**Input:** verified Azure deployment and `download_client_profiles=True`. **Output:** original package plus Windows/macOS Entra XML profiles with the resolver IP and explicit primary/hub split-tunnel routes. Temporary download URLs are not displayed or saved. Generated profiles stay local; do not commit them.

On **Windows**, import **`azurevpnconfig-windows-0.xml` from this notebook's `client` directory**. On **macOS**, import `azurevpnconfig-macos-0.xml`. Do **not** import `azurevpnconfig-original-0.xml` or the unmodified XML from `vpn-client.zip`: those have no custom resolver/suffix configuration. Each generated profile has a distinct connection name containing its **OS and deployed DNS IP**; connect that entry, not the original VNet-only name or an older DNS address. Choose **Import** in Azure VPN Client, save, then **disconnect the old connection and connect the newly named entry**. Importing XML alone does not update an established connection. Sign in to Entra. Tenant policy/Conditional Access can require additional approval. Use enterprise app assignment/custom audience/Conditional Access to govern eligible users; this notebook does not create identity policies or grant application access.

On **Linux**, stop: Entra-only Azure VPN Gateway P2S is unsupported after the Linux client retirement. Do not try to import this Entra XML into generic OpenVPN. A separately approved certificate/RADIUS design is needed.

After any peering/address/DNS change, regenerate and reimport the profile, then disconnect/reconnect. On macOS suffix entries are intentionally omitted. Validate actual per-domain OS resolution; if organizational split-DNS policies require it, use an approved macOS resolver/MDM configuration. Do not change global corporate DNS blindly.


In [ ]:
profiles = []
with utils.notebook_step('Generate Windows/macOS client profiles'):
    if not download_client_profiles or not azure_verified:
        utils.print_warning('PENDING: verify Azure resources and enable download_client_profiles to generate profiles.')
    else:
        dns_server_ip = vpn.client_dns_server(outputs, external_dns_server_ip)
        profile_outputs = {name: dict(value) for name, value in outputs.items()}
        profile_outputs['dnsResolverInboundIp']['value'] = dns_server_ip
        profiles = vpn.download_profiles(subscription_id, profile_outputs, artifact_dir / 'client', dns_suffixes, [hub_address_prefix, *primary_vnet['addressSpace']['addressPrefixes']])
        utils.print_ok(f'Profiles use deployed DNS {dns_server_ip}. Import the Windows/macOS XML for your OS, then disconnect and reconnect.', [str(path) for path in profiles])


<a id='8'></a>
### 8️⃣ Validate From the Developer Machine

**Input:** connected desktop VPN and `run_local_validation=True`. Targets default to actual primary private endpoint DNS/NIC settings and internal APIM gateway/private IPs. **Output:** direct resolver answers, effective Windows DNS/NRPT policy check, effective OS DNS restricted to expected private IPs, TCP 443 and a trusted TLS handshake with the service hostname/SNI. No insecure TLS bypass or fallback to public endpoints is used.

Leave `validation_targets=None` for discovery, or override it with `[{'host': '<service FQDN>', 'expected_ips': ['<actual private IP>']}]`. Review the discovered HTTPS inventory: only services with primary-VNet private IPs and published DNS metadata can be inferred; add any others explicitly. Redis is reported separately rather than incorrectly probed on HTTPS 443; test it using its service-specific client/port. A passing APIM check does not prove all accelerator resources. ARM lists can help: `az network private-endpoint list --resource-group <primary-rg>` and `az apim show --resource-group <primary-rg> --name <apim-name>`. Services without private endpoints/internal injection cannot be made private by the VPN alone.

| Surface | Diagnostics / acceptance |
|---|---|
| Windows terminal | `Get-DnsClientNrptPolicy -Effective`; `Resolve-DnsName <fqdn>`; `Test-NetConnection <fqdn> -Port 443`; `route print` |
| macOS terminal | `scutil --dns`; `dscacheutil -q host -a name <fqdn>`; `route -n get <private-ip>`; `nc -vz <fqdn> 443` |
| Direct DNS | `nslookup <fqdn> <resolver-ip>`; direct lookup alone does not prove effective OS/browser DNS |
| VS Code notebook | Run this cell using a local kernel on the same connected machine |
| Browser | Open the service's **original HTTPS FQDN**, not its private IP. A 401/403 may mean application authorization, not routing failure. Check DevTools remote address is the expected private IP, trusted TLS, and the intended authenticated service action succeeds |

An explicit DNS query or `nslookup` default-server result may not reflect Windows NRPT. **If direct DNS passes but OS DNS fails:** reimport the generated Windows profile from Step 7, disconnect/reconnect, and check `Get-DnsClientNrptPolicy -Effective` includes the service suffix with the deployed resolver as NameServers. If an earlier failed lookup remains cached after DNS rules are correct, run `Clear-DnsClientCache` manually and retry. A more-specific managed DNS policy can override a VPN suffix; have the network administrator resolve it instead of altering corporate policy here. Browser secure DNS/DoH, proxies and remote notebook kernels can also bypass host VPN DNS. This notebook cannot automate a browser sign-in or claim application permissions from a successful TLS connection. No P2S client pool should be added to public firewall allowlists as a substitute for private access.


In [ ]:
local_results = []
with utils.notebook_step('Validate private connectivity from this machine'):
    if not run_local_validation or not azure_verified:
        utils.print_warning('PENDING: connect Azure VPN Client and enable run_local_validation; targets are discovered automatically unless overridden.')
    else:
        targets = vpn.discover_targets(subscription_id, governance_hub_resource_group, primary_vnet, apim_name) if validation_targets is None else validation_targets
        vpn.require_values(validation_targets=targets)
        primary_networks = [ipaddress.ip_network(cidr) for cidr in primary_vnet['addressSpace']['addressPrefixes']]
        for target in targets:
            if any(not any(ipaddress.ip_address(ip) in network for network in primary_networks) for ip in target['expected_ips']):
                raise ValueError(f"{target['host']}: expected IPs must belong to the actual primary VNet address space.")
        dns_server_ip = vpn.client_dns_server(outputs, external_dns_server_ip)
        vpn.validate_resolver(dns_server_ip, targets)
        vpn.validate_windows_dns_policy(dns_server_ip, targets)
        local_results = vpn.validate_targets(targets)
        (artifact_dir / 'local-results.json').write_text(json.dumps(local_results, indent=2), encoding='utf-8')


## 📊 Results Summary

The summary distinguishes Azure provisioning from machine connectivity. Pending/disabled checks are never reported as passed. Browser application access and data-plane authorization remain a separate manual acceptance step.


In [ ]:
utils.print_info('Developer VPN validation summary')
utils.format_output('Azure resources and transit', status='PASS' if azure_verified else 'SKIP')
utils.format_output('Client profiles generated this session', [str(path) for path in profiles], status='PASS' if profiles else 'SKIP')
utils.format_output('Local direct DNS + OS DNS + TCP/TLS', local_results, status='PASS' if local_results else 'SKIP')
utils.print_warning('Browser acceptance and service-specific RBAC/API authentication require your explicit validation. This notebook does not grant those permissions.')


<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

Disconnect the desktop VPN first. Set `cleanup_enabled=True` and `cleanup_confirmation` to the exact dedicated VPN RG name. The helper validates ownership, removes only this deployment's DNS links and primary-side peering, then deletes the dedicated VPN RG. It never deletes primary private DNS zones/records or primary workloads. Gateway deletion can take a long time; keep the kernel alive and inspect failures before retrying.

Remove imported profiles from your desktop VPN client separately. Local generated artifacts remain available for troubleshooting. A partially failed deployment may need manual cleanup using its ARM operations; do not guess resource ownership or delete the primary RG.


In [ ]:
cleanup_enabled = False
cleanup_confirmation = 'REPLACE'
with utils.notebook_step('Optional VPN cleanup'):
    if not cleanup_enabled:
        utils.print_info('Cleanup disabled; persistent VPN resources continue to incur charges.')
    elif outputs is None:
        raise ValueError('Load the owned deployment outputs before cleanup.')
    else:
        vpn.cleanup(subscription_id, primary_vnet_id, outputs, cleanup_confirmation)
        outputs = None
        azure_verified = False
        local_results = []
